In [1]:
from langchain_mistralai import ChatMistralAI
from langchain_community.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests

C:\Users\hardi_cezwich\AppData\Local\Temp\ipykernel_21396\3903712936.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import tool


In [2]:
@tool
def get_conversion_factor(base_currency:str, target_currency:str)->float:
    """
    This function fetches the currency conversion between a given base currency and a target currency.
    """
    url="https://v6.exchangerate-api.com/v6/b631a524e40373f1bdb5a39a/pair/USD/INR"

    response=requests.get(url)

    return response.json()

@tool
def convert(base_currency_value: int, conversion_rate:Annotated[float, InjectedToolArg])->float:
    """
    Given a currency rate this function calculates the target currency value from a given base currency value.
    """

    return base_currency_value*conversion_rate

In [3]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1784851201,
 'time_last_update_utc': 'Fri, 24 Jul 2026 00:00:01 +0000',
 'time_next_update_unix': 1784937601,
 'time_next_update_utc': 'Sat, 25 Jul 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 96.7306}

In [4]:
convert.invoke({'base_currency_value':10, 'conversion_rate': 85.16})

851.5999999999999

In [5]:
llm=ChatMistralAI()

In [6]:
llm_with_tools=llm.bind_tools([get_conversion_factor, convert])

In [7]:
massage=[HumanMessage("What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr", additional_kwargs={}, response_metadata={})]

In [8]:
ai_massage=llm_with_tools.invoke(massage)

In [9]:
massage.append(ai_massage)

In [10]:
ai_massage.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'MnEXG2Y96',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': '8BgwdSeAX',
  'type': 'tool_call'}]

In [11]:
import json

for tool_call in ai_massage.tool_calls:
    if tool_call['name']=='get_conversion_factor':
        tool_massage1=get_conversion_factor.invoke(tool_call)
        print(tool_massage1)
        conversion_rate=json.loads(tool_massage1.content)['conversion_rate']
        massage.append(tool_massage1)
    if tool_call['name']=='convert':
        tool_call['args']['conversion_rate']=conversion_rate
        tool_massage2=convert.invoke(tool_call)
        massage.append(tool_massage2)

content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1784851201, "time_last_update_utc": "Fri, 24 Jul 2026 00:00:01 +0000", "time_next_update_unix": 1784937601, "time_next_update_utc": "Sat, 25 Jul 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 96.7306}' name='get_conversion_factor' tool_call_id='MnEXG2Y96'


In [12]:
massage

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'MnEXG2Y96', 'type': 'function', 'function': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "INR"}'}, 'index': 0}, {'id': '8BgwdSeAX', 'type': 'function', 'function': {'name': 'convert', 'arguments': '{"base_currency_value": 10}'}, 'index': 1}]}, response_metadata={'token_usage': {'prompt_tokens': 198, 'total_tokens': 232, 'completion_tokens': 34, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small', 'model': 'mistral-small', 'finish_reason': 'tool_calls'}, id='lc_run--019f9476-845c-7bc0-ad9b-2200266ba073-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'MnEXG2Y96', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_

In [13]:
llm_with_tools.invoke(massage).content

'The conversion factor between **USD** and **INR** is **1 USD = 96.7306 INR**.\n\nBased on this conversion rate, **10 USD** is equivalent to **967.31 INR** (rounded to two decimal places).'